In [88]:
from neuron import n
from neuron.units import um, mV, ms

In [89]:
#define the cell with 1 axon
axon = n.Section("axon")

In [90]:
axon.psection()

{'point_processes': {},
 'density_mechs': {},
 'ions': {},
 'morphology': {'L': 100.0,
  'diam': [500.0],
  'pts3d': [],
  'parent': None,
  'trueparent': None},
 'nseg': 1,
 'Ra': 35.4,
 'cm': [1.0],
 'regions': set(),
 'species': set(),
 'name': 'axon',
 'hoc_internal_name': '__nrnsec_0x146c4c80',
 'cell': None}

In [91]:
#set axon length and diameter 
axon.diam = 238*um #um, diam of giant squid axon
axon.L = 100000*um #um, length of axon
axon.nseg = 100 #number of segments in axon

In [92]:
#set axial resistance and membrane capacitance
axon.Ra = 35.4 #Ohm-cm axial resistance
axon.cm = 1 #uF/cm2 membrane capacitance

In [93]:
#insert hodgkin-huxley model
axon.insert(n.hh)
for seg in axon:
    seg.hh.gnabar = 0.12 #S/cm2 sodium conductance
    seg.hh.gkbar = 0.036 #S/cm2 potassium conductance
    seg.hh.gl = 0.0003 #S/cm2 leak conductance
    
    seg.ena = 50*mV #mV sodium reverse potential
    seg.ek = -77*mV #mV potassium reverse potential
    seg.hh.el = -54.4*mV #mV leak reverse potential

In [94]:
#stimulus - current clamp
stim = n.IClamp(axon(0)) #current clamp at the 0th segment
stim.delay = 5 #ms
stim.dur = 95 #ms
stim.amp = 2000 #nA

In [95]:
# axon.psection()

In [96]:
#record variables
v_segs = []
m_segs = []
h_segs = []
n_segs = []

for seg in axon:
    v_seg = n.Vector().record(seg._ref_v) #record voltage at center of every segment
    m_seg = n.Vector().record(seg.hh._ref_m)
    h_seg = n.Vector().record(seg.hh._ref_h)
    n_seg = n.Vector().record(seg.hh._ref_n)
    v_segs.append(v_seg)
    m_segs.append(m_seg)
    h_segs.append(h_seg)
    n_segs.append(n_seg)


t = n.Vector().record(n._ref_t) #record timestamp vector

In [97]:
#initialize and run
n.load_file("stdrun.hoc")

n.dt = 0.1 # time step in ms | default is 0.025ms
n.steps_per_ms = 1 / n.dt  # keep the standard run system in sync

n.finitialize(-65*mV) #this initializes the resting values for m, h, and n gates as well
n.continuerun(100*ms)

0.0

## Results

In [98]:
from bokeh.io import output_notebook
import bokeh.plotting as plt
from bokeh.palettes import Viridis256
import numpy as np
output_notebook()

Loading BokehJS ...

In [99]:
f = plt.figure(x_axis_label="time (ms)", y_axis_label="v (mV)", title="membrane potential vs time")
N = len(v_segs)
# for i, (seg, v) in enumerate(zip(axon, v_segs)):
#     color = Viridis256[int(i * 255 / max(N - 1, 1))]
#     f.line(np.array(t), np.array(v), color=color, legend_label=f"x = {seg.x:.3f}")

f.line(np.array(t), np.array(v_segs[0]), line_width=2)

# f.legend.click_policy = "hide"   # click a legend entry to hide/show that line
plt.show(f)

In [100]:
f2 = plt.figure(x_axis_label="v (mV)", y_axis_label="n", title="limit cycle")
f2.line(np.array(v_segs[0]), np.array(n_segs[0]))
plt.show(f2)

In [101]:
print(n.units("IClamp.amp"))

nA


In [102]:
#function to create vtu/vtk files
import os
import numpy as np

def segment_coords(sec):
    """3D coordinates (µm) of each segment center in a section."""
    n_pts = sec.n3d()
    seg_x = np.array([seg.x for seg in sec])
    if n_pts < 2:                                   # no 3D geometry: lay it out along the x-axis
        return np.column_stack([seg_x * sec.L, np.zeros_like(seg_x), np.zeros_like(seg_x)])
    arc = np.array([sec.arc3d(i) for i in range(n_pts)])
    xyz = np.array([[sec.x3d(i), sec.y3d(i), sec.z3d(i)] for i in range(n_pts)])
    target = seg_x * arc[-1]
    return np.column_stack([np.interp(target, arc, xyz[:, k]) for k in range(3)])


def _fmt(arr):
    return " ".join(f"{x:.6g}" for x in np.ravel(arr))

def write_vtk_series(sec, t, data, out_dir=".", vtu_folder="vtu", prefix="axon", every=1):
    vtu_dir = os.path.join(out_dir, vtu_folder)
    os.makedirs(vtu_dir, exist_ok=True)
    """
    sec    : the NEURON section (e.g. axon)
    t      : recorded time Vector
    data   : dict of name -> list of recorded Vectors, one per segment
             e.g. {"v": v_segs, "m": m_segs, "h": h_segs, "n": n_segs}
    every  : write every k-th time step (use >1 to keep file count down)
    """

    pts = segment_coords(sec)
    n_points = len(pts)
    diam = np.array([seg.diam for seg in sec])
    T = np.array(t)
    arrays = {name: np.array([np.array(v) for v in vecs]) for name, vecs in data.items()}  # (nseg, nt)

    # cells: lines joining neighbouring segment centers (a single vertex if nseg = 1)
    if n_points > 1:
        conn = np.column_stack([np.arange(n_points - 1), np.arange(1, n_points)])
        offsets = np.arange(2, 2 * (n_points - 1) + 1, 2)
        types = np.full(n_points - 1, 3)            # VTK_LINE
    else:
        conn, offsets, types = np.array([0]), np.array([1]), np.array([1])   # VTK_VERTEX
    n_cells = len(types)

    pvd_entries = []
    for frame, i in enumerate(range(0, len(T), every)):
        fname = f"{prefix}_{frame:05d}.vtu"
        point_data = "".join(
            f'        <DataArray type="Float64" Name="{name}" format="ascii">{_fmt(arr[:, i])}</DataArray>\n'
            for name, arr in arrays.items()
        )
        point_data += f'        <DataArray type="Float64" Name="diam" format="ascii">{_fmt(diam)}</DataArray>\n'

        vtu = f"""<?xml version="1.0"?>
<VTKFile type="UnstructuredGrid" version="0.1" byte_order="LittleEndian">
  <UnstructuredGrid>
    <Piece NumberOfPoints="{n_points}" NumberOfCells="{n_cells}">
      <Points>
        <DataArray type="Float64" NumberOfComponents="3" format="ascii">{_fmt(pts)}</DataArray>
      </Points>
      <Cells>
        <DataArray type="Int32" Name="connectivity" format="ascii">{_fmt(conn)}</DataArray>
        <DataArray type="Int32" Name="offsets" format="ascii">{_fmt(offsets)}</DataArray>
        <DataArray type="UInt8" Name="types" format="ascii">{_fmt(types)}</DataArray>
      </Cells>
      <PointData Scalars="v">
{point_data}      </PointData>
    </Piece>
  </UnstructuredGrid>
</VTKFile>
"""

        with open(os.path.join(vtu_dir, fname), "w") as fh:
            fh.write(vtu)
        pvd_entries.append(
            f'    <DataSet timestep="{T[i]:.6g}" group="" part="0" file="{vtu_folder}/{fname}"/>'
        )

    pvd = ('<?xml version="1.0"?>\n'
           '<VTKFile type="Collection" version="0.1" byte_order="LittleEndian">\n'
           '  <Collection>\n' + "\n".join(pvd_entries) + "\n  </Collection>\n</VTKFile>\n")
    pvd_path = os.path.join(out_dir, f"{prefix}.pvd")
    with open(pvd_path, "w") as fh:
        fh.write(pvd)
    return pvd_path

In [103]:
pvd = write_vtk_series(axon, t, {"v": v_segs, "m": m_segs, "h": h_segs, "n": n_segs},
                 out_dir="results", vtu_folder="vtu", every=1)
print(pvd)

results/axon.pvd
